In [1]:
"""Lab 2, Section 2.5 - reflex classroom agent (no memory).
Run: python lab2_percepts.py
Modes ECO/COOL/WARM/IDLE are simulated labels; no hardware is controlled.
Empty room has priority over temperature. Exactly 20 or 26 C occupied -> IDLE.
"""

class ClassroomAgent:
    def act(self, temp, occupied):
        if not occupied:
            return "ECO"
        if temp > 26:
            return "COOL"
        if temp < 20:
            return "WARM"
        return "IDLE"

agent = ClassroomAgent()
percepts = [(29, True), (29, True), (20, True),
            (26, True), (19, True), (29, False)]
for step, percept in enumerate(percepts, 1):
    print(step, percept, "target mode", agent.act(*percept))

1 (29, True) target mode COOL
2 (29, True) target mode COOL
3 (20, True) target mode IDLE
4 (26, True) target mode IDLE
5 (19, True) target mode WARM
6 (29, False) target mode ECO


In [2]:
"""Lab 2, Task 3 - stateful classroom agent.
Remembers the previous target mode and reports whether a command must be sent.
Thresholds and occupancy priority are unchanged from the reflex agent.
Run: python lab2_model.py
"""

class StatefulClassroomAgent:
    def __init__(self):
        self.previous_mode = None          # internal storage; None = nothing sent yet

    def decide(self, temp, occupied):
        """Same policy as the reflex agent (occupancy first, then 26 / 20 C)."""
        if not occupied:
            return "ECO"
        if temp > 26:
            return "COOL"
        if temp < 20:
            return "WARM"
        return "IDLE"

    def act(self, temp, occupied):
        """Return (target_mode, send_command). Command is sent only on a change."""
        target = self.decide(temp, occupied)
        send = target != self.previous_mode
        self.previous_mode = target
        return target, send


if __name__ == "__main__":
    agent = StatefulClassroomAgent()
    # repeated percepts, both exact boundaries (20, 26), mode transition, empty room
    percepts = [(29, True), (29, True), (20, True),
                (26, True), (19, True), (29, False)]
    print("step | previous | percept | target | command")
    for step, percept in enumerate(percepts, 1):
        prev = agent.previous_mode
        target, send = agent.act(*percept)
        print(step, prev, percept, target, "SEND" if send else "no-command", sep=" | ")

step | previous | percept | target | command
1 | None | (29, True) | COOL | SEND
2 | COOL | (29, True) | COOL | no-command
3 | COOL | (20, True) | IDLE | SEND
4 | IDLE | (26, True) | IDLE | no-command
5 | IDLE | (19, True) | WARM | SEND
6 | WARM | (29, False) | ECO | SEND


In [4]:
"""Lab 2 - checks, reflex/stateful comparison and the noisy-26C edge case.
Run: python lab2_check.py
Temperatures are synthetic lab values; the model is a simulator, not hardware.
"""
#from lab2_percepts import ClassroomAgent          # prints its own trace on import
#from lab2_model import StatefulClassroomAgent

print("\n=== Check 1: expected modes from the manual ===")
percepts = [(29, True), (29, True), (20, True), (26, True), (19, True), (29, False)]
expected = ["COOL", "COOL", "IDLE", "IDLE", "WARM", "ECO"]
reflex, stateful = ClassroomAgent(), StatefulClassroomAgent()
for p, e in zip(percepts, expected):
    r = reflex.act(*p)
    s, _ = stateful.act(*p)
    print(p, "expected", e, "reflex", r, "stateful", s, "OK" if r == s == e else "MISMATCH")

print("\n=== Check 2: commands sent, reflex vs stateful ===")
stateful = StatefulClassroomAgent()
reflex_cmds = len(percepts)                       # reflex re-issues a command every step
stateful_cmds = sum(send for _, send in (stateful.act(*p) for p in percepts))
print("reflex commands :", reflex_cmds)
print("stateful commands:", stateful_cmds)

print("\n=== Check 3 (edge case): noisy readings around 26 C ===")
noisy = [(25.9, True), (26.1, True), (25.9, True), (26.1, True), (25.9, True)]
stateful = StatefulClassroomAgent()
print("step | previous | percept | target | command")
n = 0
for i, p in enumerate(noisy, 1):
    prev = stateful.previous_mode
    t, send = stateful.act(*p)
    n += send
    print(i, prev, p, t, "SEND" if send else "no-command", sep=" | ")
print("commands sent:", n, "of", len(noisy), "-> memory alone gives NO hysteresis")

print("\n=== Check 4 (proposed improvement): hysteresis band ===")
class HysteresisAgent(StatefulClassroomAgent):
    COOL_ON, COOL_OFF = 26, 25          # synthetic lab values: start >26, stop <25
    def decide(self, temp, occupied):
        if not occupied:
            return "ECO"
        if temp > self.COOL_ON or (self.previous_mode == "COOL" and temp >= self.COOL_OFF):
            return "COOL"
        if temp < 20:
            return "WARM"
        return "IDLE"
h = HysteresisAgent()
n = 0
for i, p in enumerate(noisy, 1):
    prev = h.previous_mode
    t, send = h.act(*p)
    n += send
    print(i, prev, p, t, "SEND" if send else "no-command", sep=" | ")
print("commands sent:", n, "of", len(noisy))


=== Check 1: expected modes from the manual ===
(29, True) expected COOL reflex COOL stateful COOL OK
(29, True) expected COOL reflex COOL stateful COOL OK
(20, True) expected IDLE reflex IDLE stateful IDLE OK
(26, True) expected IDLE reflex IDLE stateful IDLE OK
(19, True) expected WARM reflex WARM stateful WARM OK
(29, False) expected ECO reflex ECO stateful ECO OK

=== Check 2: commands sent, reflex vs stateful ===
reflex commands : 6
stateful commands: 4

=== Check 3 (edge case): noisy readings around 26 C ===
step | previous | percept | target | command
1 | None | (25.9, True) | IDLE | SEND
2 | IDLE | (26.1, True) | COOL | SEND
3 | COOL | (25.9, True) | IDLE | SEND
4 | IDLE | (26.1, True) | COOL | SEND
5 | COOL | (25.9, True) | IDLE | SEND
commands sent: 5 of 5 -> memory alone gives NO hysteresis

=== Check 4 (proposed improvement): hysteresis band ===
1 | None | (25.9, True) | IDLE | SEND
2 | IDLE | (26.1, True) | COOL | SEND
3 | COOL | (25.9, True) | COOL | no-command
4 | COOL 